In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss
from  sklearn.ensemble import  RandomForestClassifier
from sklearn.calibration import CalibrationDisplay
from scipy.stats import poisson_binom
import scipy,sklearn

In [78]:
# spectral clustering example

In [79]:
adj = np.array([
    [0, 1, 0, 0],
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
])

d = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])


In [80]:
L = d - adj
print(L)

[[ 1 -1  0  0]
 [-1  1  0  0]
 [ 0  0  1 -1]
 [ 0  0 -1  1]]


In [81]:
res = np.linalg.eig(L)

In [82]:
print(res.eigenvalues)

[2. 0. 2. 0.]


In [83]:
print(res.eigenvectors)

[[ 0.70710678  0.70710678  0.          0.        ]
 [-0.70710678  0.70710678  0.          0.        ]
 [ 0.          0.          0.70710678  0.70710678]
 [ 0.          0.         -0.70710678  0.70710678]]


In [84]:
print(np.dot(L, res.eigenvectors.T[0]))
print(res.eigenvectors.T[0] * res.eigenvalues[0])


[ 1.41421356 -1.41421356  0.          0.        ]
[ 1.41421356 -1.41421356  0.          0.        ]


In [85]:
top_3 = [0, 2, 3]
new_points = res.eigenvectors[top_3]

In [86]:
new_points.T

array([[ 0.70710678,  0.        ,  0.        ],
       [ 0.70710678,  0.        ,  0.        ],
       [ 0.        ,  0.70710678, -0.70710678],
       [ 0.        ,  0.70710678,  0.70710678]])

In [87]:
# nodes:
#  1 - 2 - 3     4 - 5     6 - 7
#
# this is the 3 groups of nodes

adjacent = np.array([
    #1  2  3  4  5  6  7
    [0, 1, 1, 0, 0, 0, 0], # 1
    [1, 0, 1, 0, 0, 0, 0], # 2
    [1, 1, 0, 0, 0, 0, 0], # 3
    [0, 0, 0, 0, 1, 0, 0], # 4
    [0, 0, 0, 1, 0, 0, 0], # 5
    [0, 0, 0, 0, 0, 0, 1], # 6
    [0, 0, 0, 0, 0, 1, 0], # 7 
])

diag = np.array([
    #1  2  3  4  5  6  7
    [2, 0, 0, 0, 0, 0, 0], # 1
    [0, 2, 0, 0, 0, 0, 0], # 2
    [0, 0, 2, 0, 0, 0, 0], # 3
    [0, 0, 0, 1, 0, 0, 0], # 4
    [0, 0, 0, 0, 1, 0, 0], # 5
    [0, 0, 0, 0, 0, 1, 0], # 6
    [0, 0, 0, 0, 0, 0, 1], # 7
])

lap = diag - adjacent
lap

array([[ 2, -1, -1,  0,  0,  0,  0],
       [-1,  2, -1,  0,  0,  0,  0],
       [-1, -1,  2,  0,  0,  0,  0],
       [ 0,  0,  0,  1, -1,  0,  0],
       [ 0,  0,  0, -1,  1,  0,  0],
       [ 0,  0,  0,  0,  0,  1, -1],
       [ 0,  0,  0,  0,  0, -1,  1]])

In [88]:
ress = np.linalg.eig(lap)
print(ress.eigenvalues)

[ 3.0000000e+00 -5.3481844e-16  3.0000000e+00  2.0000000e+00
  0.0000000e+00  2.0000000e+00  0.0000000e+00]


In [89]:
print(ress.eigenvectors)

[[ 0.81649658 -0.57735027  0.24423262  0.          0.          0.
   0.        ]
 [-0.40824829 -0.57735027 -0.79684797  0.          0.          0.
   0.        ]
 [-0.40824829 -0.57735027  0.55261536  0.          0.          0.
   0.        ]
 [ 0.          0.          0.          0.70710678  0.70710678  0.
   0.        ]
 [ 0.          0.          0.         -0.70710678  0.70710678  0.
   0.        ]
 [ 0.          0.          0.          0.          0.          0.70710678
   0.70710678]
 [ 0.          0.          0.          0.          0.         -0.70710678
   0.70710678]]


In [90]:
sort = np.argsort(ress.eigenvalues)
sort

array([1, 4, 6, 3, 5, 0, 2])

In [91]:
smallest_3 = ress.eigenvectors[:, sort[:3]]
smallest_3

array([[-0.57735027,  0.        ,  0.        ],
       [-0.57735027,  0.        ,  0.        ],
       [-0.57735027,  0.        ,  0.        ],
       [ 0.        ,  0.70710678,  0.        ],
       [ 0.        ,  0.70710678,  0.        ],
       [ 0.        ,  0.        ,  0.70710678],
       [ 0.        ,  0.        ,  0.70710678]])

In [93]:
# checking to see if the data is zero-centered first
# points : (-1, -1)   (0.5, -0.5)   (1, 1)   (-0.5 0.5)

x = ( (-1) + 0.5 + 1 + (-0.5) ) / 4

y = ( (-1) + (-0.5) + 1 + 0.5) / 4

print(x, y)

0.0 0.0


In [95]:
from sklearn.decomposition import PCA
X = np.array([
    [-1, -1],
    [0.5, -0.5],
    [1, 1],
    [-0.5, 0.5]
])

pca = PCA(n_components = 2)
pca.fit(X)

pc1 = pca.components_[0]
print(pc1)

[0.70710678 0.70710678]


In [108]:
# now solving it manually to make sure we do this correctly

varX = (-1)**2 + (0.5)**2 + (1)**2 + (-0.5)**2

varY = (-1)**2 + (0.5)**2 + (1)**2 + (-0.5)**2

covXY = (-1)*(-1) + (0.5)*(-0.5) + (1)*(1) + (-0.5)*(0.5)

E = np.array([
    [varX, covXY],
    [covXY, varY]
])
E

array([[2.5, 1.5],
       [1.5, 2.5]])

In [109]:
matrix = (1/4) * E
matrix

array([[0.625, 0.375],
       [0.375, 0.625]])

In [111]:
# now to find the eigenvalues

# det(matrix - lambda*I) = 0

# [0.625 - lambda, 0.375]
# [0.375, 0.625 - lambda]

#  (0.625 - l)(0.625 - l) - (0.375)(0.375) = 0
#  (0.625 - l)^2 - (0.375)^2 = 0
#  (0.625 - l)^2 - 0.140625 = 0
#  (0.625 - l)^2 = 0.140625
np.sqrt(0.140625)

np.float64(0.375)

In [113]:
# (0.625 - l) = +- 0.375
# 
lambda1 = 0.625 + 0.375
lambda2 = 0.625 - 0.375
print(lambda1, lambda2)
# here, the first one is larger so we will use lambda = 1

1.0 0.25


In [117]:
#   [0.625 - 1, 0.375]  [v1]  =  [0]
#   [0.375, 0.625 - 1]  [v2]  =  [0]

#          =

#   [-0.375, 0.375]  [v1]  =  [0]
#   [0.375, -0.375]  [v2]  =  [0]

# -0.375, 0.375 = 0 = v1
# 0.375, -0.375 = 0 = v2

# v1 = v2

# ||v|| = sqrt (v1^2 + v2^2) = 1

# since they are both the same, we can rewrite as

# ||v|| = sqrt (2v1^2) = 1

# ---> 2v1^2 = 1^2

# ---> 2v1^2 = 1

# ---> v1^2 = 1/2

# ---> v1 = sqrt (1/2)

# ---> v1 = 1/(sqrt2)

v = 1/np.sqrt(2)

V = np.array([
    [v],
    [v]
])
V



array([[0.70710678],
       [0.70710678]])